# Comparison Summary · Methods and Results Side by Side

This notebook makes the four retrieval modes **easy to judge even if you have not read the source document**. It does three things:

1. States each anchor question **and a rubric** — the key points a strong answer should contain — so "better" becomes measurable instead of subjective.
2. Runs a **controlled matrix** (same questions × all four modes on one rich index) and scores each answer's **rubric coverage**.
3. Prints the **full answers side by side** (no truncation) so you can read the actual detail each mode returned, then loads every recorded lab run for the whole workshop.

**Anchor questions**
- **Q1 (precise / keyword):** site-investigation objectives — favours full-text.
- **Q2 (conceptual / multi-hop):** groundwater + clay risks and design — favours vector / hybrid / agentic.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Use the richest index for a fair cross-mode comparison

In [2]:
job = lab.ingest(skill_profile='visual_nlp', reuse=True)
lab.chunk_overview(job)

Reusing existing 'visual_nlp' ingestion (doc_id=edbcb680, 414 chunks). Pass reuse=False to force a fresh run.


{'doc_id': 'edbcb680033d4886af1b6cfab59ccddc',
 'skill_profile': 'visual_nlp',
 'chunk_count': 414,
 'avg_tokens': 208.4,
 'max_tokens': 1976,
 'chunks_with_summary': 414,
 'chunks_with_keyword_hints': 414,
 'chunks_with_image_description': 0}

## Step 3 — Define the questions and a scoring rubric

The hard part of comparing retrieval modes is knowing what a *good* answer looks like. So for each question we declare a **rubric**: the concrete key points a complete answer should mention. Each key point is matched against the answer text (case-insensitive), which turns a vague "this reads better" into a concrete **coverage score** like `6/8`.

The rubrics below are grounded in the deep-excavation source material, but you do not need to know that document to read the results — the rubric tells you what to look for.

In [3]:
from IPython.display import Markdown, display

Q1 = 'What are the objectives of site investigation for ELS works?'
Q2 = 'Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?'

# Rubric = the key points a strong answer should mention. Each entry is
# (label, [match terms]); a point is 'covered' if any term appears in the answer.
QUESTIONS = {
    'Q1 precise': {
        'text': Q1,
        'asks_for': 'A checklist of the standard site-investigation activities for excavation and lateral support (ELS) works.',
        'keypoints': [
            ('Desk study', ['desk study']),
            ('Site reconnaissance', ['reconnaissance']),
            ('Topographic survey', ['topographic', 'topographical']),
            ('Ground investigation', ['ground investigation']),
            ('Groundwater regime', ['groundwater', 'water table']),
            ('Nearby buildings / services', ['building', 'structure', 'services', 'utilit']),
            ('Safe execution', ['safe', 'safety']),
        ],
    },
    'Q2 multi-hop': {
        'text': Q2,
        'asks_for': 'Reasoning that links high groundwater and soft clay to specific excavation risks and the design measures that mitigate them.',
        'keypoints': [
            ('Base heave / basal stability', ['base heave', 'basal', 'heave']),
            ('Piping / boiling / uplift', ['piping', 'boiling', 'uplift', 'hydraulic']),
            ('Groundwater control / dewatering', ['dewater', 'groundwater control', 'drawdown', 'cut-off', 'cutoff']),
            ('Ground settlement', ['settlement']),
            ('Wall deflection / movement', ['deflection', 'lateral movement', 'wall movement']),
            ('Soft clay / undrained strength', ['undrained', 'soft clay', 'shear strength']),
            ('Support system (struts / anchors)', ['strut', 'anchor', 'support system']),
            ('Monitoring / instrumentation', ['monitor', 'instrumentation']),
        ],
    },
}

MODES = ['full_text', 'vector', 'hybrid', 'agentic']

def show_mode_answers(qlabel, *, job, modes=MODES):
    """Render the FULL answer from each retrieval mode with rubric coverage."""
    spec = QUESTIONS[qlabel]
    kps = spec['keypoints']
    display(Markdown(
        f"### {qlabel} — *{spec['text']}*\n\n"
        f"**What this question asks for:** {spec['asks_for']}\n\n"
        f"**Rubric (a strong answer should mention):** "
        + ', '.join(label for label, _ in kps)
    ))
    for mode in modes:
        resp = lab.ask(spec['text'], job=job, retrieval_mode=mode)
        cov = lab.keypoint_coverage(resp.answer, kps)
        block = (
            f"#### `{mode}` — coverage **{cov['score']}** · "
            f"{len(resp.citations)} citations · {len(resp.answer)} chars\n\n"
            f"{resp.answer.strip()}\n\n"
        )
        if cov['missing']:
            block += f"> **Not mentioned:** {', '.join(cov['missing'])}\n\n"
        block += f"**Top citations:** {lab.citation_summary(resp, top=3)}"
        display(Markdown(block))
        display(Markdown('---'))

## Step 4 — Controlled matrix: coverage at a glance

Each row is a full grounded chat turn (retrieval + synthesis), the same as the UI. The **coverage** column scores the answer against the rubric, so you can compare modes without reading every answer yet.

In [4]:
import pandas as pd

matrix = []
for qlabel, spec in QUESTIONS.items():
    for mode in MODES:
        resp = lab.ask(spec['text'], job=job, retrieval_mode=mode,
                       record_as=f"compare_{mode}_{qlabel.split()[0]}")
        cov = lab.keypoint_coverage(resp.answer, spec['keypoints'])
        matrix.append({
            'question': qlabel,
            'mode': mode,
            'coverage': cov['score'],
            'citations': len(resp.citations),
            'answer_chars': len(resp.answer),
            'top_section': lab.citation_summary(resp, top=1),
        })
matrix_df = pd.DataFrame(matrix)
matrix_df

,question,mode,coverage,citations,answer_chars,top_section
0,Q1 precise,full_text,5/7,8,1080,2.1 Site Investigation p.[17]
1,Q1 precise,vector,5/7,8,1063,2.1 Site Investigation p.[17]
2,Q1 precise,hybrid,5/7,7,1001,2.1 Site Investigation p.[17]
3,Q1 precise,agentic,7/7,8,760,2.1 Site Investigation p.[17]
4,Q2 multi-hop,full_text,1/8,8,1005,10.2.3 Groundwater Monitoring p.[127]
5,Q2 multi-hop,vector,1/8,8,987,5.2 Site Conditions Particularly Vulnerable to...
6,Q2 multi-hop,hybrid,3/8,8,1041,7.2 Overall Stability p.[76]
7,Q2 multi-hop,agentic,8/8,8,3554,6.5.1 Design Groundwater Level for Ultimate Li...


### Read the matrix

Higher **coverage** with fewer wasted citations is the signal to watch. Expect full-text to do well on Q1's exact terminology, and vector / hybrid / agentic to lift coverage on the multi-hop Q2 where the answer must connect several sections. The next two steps show the **full answers** so you can confirm the scores by reading.

## Step 5 — Full answers side by side · Q1 (precise)

Complete answers, no truncation, each tagged with its rubric coverage and the points it missed.

In [5]:
show_mode_answers('Q1 precise', job=job)

### Q1 precise — *What are the objectives of site investigation for ELS works?*

**What this question asks for:** A checklist of the standard site-investigation activities for excavation and lateral support (ELS) works.

**Rubric (a strong answer should mention):** Desk study, Site reconnaissance, Topographic survey, Ground investigation, Groundwater regime, Nearby buildings / services, Safe execution

#### `full_text` — coverage **5/7** · 8 citations · 1080 chars

The main objectives of site investigation are to acquire knowledge of site characteristics that affect such works and plan for their safe execution, with due consideration given to the nearby buildings/structures/services. When planning ELS works, site investigation should normally include a detailed desk study, site reconnaissance, topographic survey, groun [1]

Supporting evidence:
- The various stages of site investigation, design and construction of ELS works require coordinated input from experienced designers and contractors. Continuous involvement of the designer of the ELS works is essential for verifying both the validity of the assumed ground conditions and the expected performance of the ELS works. [8]
- Good practice for site investigation and selection of geotechnical parameters that are crucial for the design of ELS works and associated key considerations are presented in Chapter 2. A review of common types of excavation support systems and technical considerations for the design and construction of ELS works are discussed in Chapters 3, 4 and 5. [7]

> **Not mentioned:** Ground investigation, Groundwater regime

**Top citations:** 2.1 Site Investigation p.[17]; 1.3 General Guidance p.[16]; 1.2 Overview p.[15, 16]

---

#### `vector` — coverage **5/7** · 8 citations · 1063 chars

The main objectives of site investigation are to acquire knowledge of site characteristics that affect such works and plan for their safe execution, with due consideration given to the nearby buildings/structures/services. When planning ELS works, site investigation should normally include a detailed desk study, site reconnaissance, topographic survey, groun [1]

Supporting evidence:
- GI should be properly planned in order to develop a suitable ground model for the design of ELS works, with due regard to the anticipated extent and depth of excavation. An adequate number of boreholes should be extended to the competent soil stratum or bedrock when determining the ground stratigraphy. [2]
- Guidance on the planning of monitoring for the design of retaining walls, as given in Geoguide 1, is also generally applicable for ELS works. A site-specific I&M plan should be formulated and should include the following general requirements: (a) Monitoring checkpoints for sensitive receivers located within the influence zones of the construction site (espec [4]

> **Not mentioned:** Ground investigation, Groundwater regime

**Top citations:** 2.1 Site Investigation p.[17]; 2.1.1 Ground Investigation p.[17]; 10.2.1 General Requirements p.[126]

---

#### `hybrid` — coverage **5/7** · 7 citations · 1001 chars

The main objectives of site investigation are to acquire knowledge of site characteristics that affect such works and plan for their safe execution, with due consideration given to the nearby buildings/structures/services. When planning ELS works, site investigation should normally include a detailed desk study, site reconnaissance, topographic survey, groun [1]

Supporting evidence:
- Action Level 3 is intended to flag up the situation where the ELS works are under-performing when compared with other ELS works in similar ground conditions and site settings. The trigger value of Action Level 3 should be set to 0.5%He, and subject to a maximum empirical value of 100 mm. [5]
- any significant damage of subgrade and services, potential sinkholes), arrange a joint site inspection with relevant authorities and submit the investigation report to the PM1 or the AP/RSE/RGE2 and the relevant authorities for acceptance. The Contractor shall propose and implement necessary remedial works (e.g. [4]

> **Not mentioned:** Ground investigation, Groundwater regime

**Top citations:** 2.1 Site Investigation p.[17]; 9.2.2.3 Empirical Limits for Ground Settlement p.[114]; Response Actions p.[120]

---

#### `agentic` — coverage **7/7** · 8 citations · 595 chars

The objectives of site investigation for ELS works are to: - acquire knowledge of site characteristics that affect the ELS works; and - plan for their safe execution, with due consideration to nearby buildings, structures, and services. [1] For planning ELS works, the site investigation should normally include: - a detailed desk study; - site reconnaissance; - topographic survey; - ground investigation to establish a suitable ground model; - collection of soil and rock samples; - in-situ tests for selecting design parameters; and - field monitoring to determine groundwater conditions. [1]

**Top citations:** 2.1 Site Investigation p.[17]; 2.1.2 Groundwater Conditions p.[17, 18]; Foreword p.[4]

---

## Step 6 — Full answers side by side · Q2 (multi-hop)

This is where the qualitative gap is clearest: watch how the conceptual modes cover more of the risk/design rubric than plain keyword search.

In [6]:
show_mode_answers('Q2 multi-hop', job=job)

### Q2 multi-hop — *Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?*

**What this question asks for:** Reasoning that links high groundwater and soft clay to specific excavation risks and the design measures that mitigate them.

**Rubric (a strong answer should mention):** Base heave / basal stability, Piping / boiling / uplift, Groundwater control / dewatering, Ground settlement, Wall deflection / movement, Soft clay / undrained strength, Support system (struts / anchors), Monitoring / instrumentation

#### `full_text` — coverage **1/8** · 8 citations · 1005 chars

However, their locations, length and positioning of response zones should be properly planned with due consideration given to the hydrogeological condition of the groundwater regime adjacent to the site. Where the ground conditions comprise stratified marine clay underlain by sandy soils (e.g. [5]

Supporting evidence:
- Given the temporary nature of ELS works, DGWL should be related to possible scenarios that could occur within the duration of the works for different limit states. It is not necessary to consider the effects of long-term and extreme events (e.g. due to climate change). Based on local experience and practice, the following considerations are usually adopted i [6]
- The design high groundwater level (DHGWL) for SLS should be based on realistic estimation of the highest groundwater level and used for assessing wall deflection caused by the excavation. On the contrary, the design low groundwater level (DLGWL) for SLS should be based on the lowest recorded groundwater level. [4]

> **Not mentioned:** Base heave / basal stability, Piping / boiling / uplift, Groundwater control / dewatering, Ground settlement, Soft clay / undrained strength, Support system (struts / anchors), Monitoring / instrumentation

**Top citations:** 10.2.3 Groundwater Monitoring p.[127]; 6.5 Design Groundwater Level p.[69]; 6.5.2 Design Groundwater Level for Serviceability Limit State p.[71]

---

#### `vector` — coverage **1/8** · 8 citations · 987 chars

The presence of loose fill renders a site more susceptible to excessive ground loss during piling operations and bulk excavation, particularly where the groundwater table is near the ground surface. This is commonly the case in reclaimed land where the fill was placed by the hydraulic filling method. [2]

Supporting evidence:
- ULS Design (0.5 m of groundwater (a) Inspect and examine the performance of the ELS works and the response of nearby sensitive receivers with respect to their structural stability and serviceability. level below the highest DGWL) (b) Investigate causes and any correlations with observed changes in groundwater levels. [4]
- Site-specific groundwater and drainage conditions should be ascertained within and in the vicinity of the site, and their likely response to, for example, storms, seasonal rise, artesian conditions or tidal action. Field tests normally yield more reliable parameters of the mass permeability of the ground than laboratory tests. [1]

> **Not mentioned:** Base heave / basal stability, Groundwater control / dewatering, Ground settlement, Wall deflection / movement, Soft clay / undrained strength, Support system (struts / anchors), Monitoring / instrumentation

**Top citations:** 5.2 Site Conditions Particularly Vulnerable to Ground Loss p.[53]; Response Actions p.[122]; 2.1.2 Groundwater Conditions p.[17, 18]

---

#### `hybrid` — coverage **3/8** · 8 citations · 1041 chars

GEO Publication No. 1/2023, **Deep Excavation Design and Construction**, is a Hong Kong Geotechnical Engineering Office publication first published in December 2023. Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is. [5]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and should be aimed to determine site-specific  [4]
- As discussed in Section 2.1.2, the differential piezometric pressures between marine clay and alluvial sand layers could lead to porewater pressure dissipation and associated consolidation settlement in the clay layer even when the groundwater level in an overlying fill layer remains stable. [6]

> **Not mentioned:** Base heave / basal stability, Piping / boiling / uplift, Groundwater control / dewatering, Wall deflection / movement, Support system (struts / anchors)

**Top citations:** 7.2 Overall Stability p.[76]; Ring bit p.[54, 55]; 8.3.3.2 Consolidation p.[104, 105]

---

#### `agentic` — coverage **8/8** · 8 citations · 2757 chars

- Key excavation risks with **high groundwater** and **clay layers** include: - **Groundwater-related instability**, because groundwater pressure can govern wall forces and ground deformation. [1][2] - **Excessive drawdown and consolidation settlement** in low-permeability clay, especially marine clay, where delayed porewater-pressure dissipation can cause settlement long after excavation starts. [3][4] - **Uplift / base heave / hydraulic failure**, especially where soft clay extends below the excavation base or where artesian pressure beneath a low-permeability layer exceeds the overburden pressure. [5][6] - **Piping or seepage failure**, where upward seepage reduces effective stress and can wash out soil, potentially causing sinkholes; this is more likely in loose, permeable sandy layers. [7] - **Overall stability problems**, especially near steeply sloping ground, or where weak layers such as loose sand or soft clay exist below the embedded wall toe. [8] - **Unexpected groundwater pressure conditions**, because clayey or layered ground may create confined aquifers or perched water tables that differ from simple hydrostatic assumptions.[4] - Key design considerations include: - **Thorough site investigation** to establish stratigraphy, permeability, groundwater conditions, and any confined aquifers; suitable piezometers/standpipes should be installed at appropriate depths.[4] - **Careful selection of the design groundwater level** for ULS and SLS, including allowances for seasonal variation, tidal effects, and potential damming effects from the excavation support system.[2] - **Water cut-off and seepage control**, such as embedded walls and grout curtains, with seepage analysis or flow-net analysis where groundwater inflow cannot be ignored. [2] - **Sufficient wall embedment** into firmer/less permeable strata to reduce seepage, prevent piping, and avoid base heave. [7][5] - **Monitoring and trigger/action levels** for groundwater and settlement, with contingency measures such as regrouting or recharge wells if groundwater rises or drawdown becomes excessive. - **Construction sequence and buildability**, including staged excavation, timely strut installation, and minimizing excessive dewatering to reduce wall deflection and settlement. - **Numerical or limit-equilibrium analysis as appropriate**, with more complex ground conditions often needing numerical methods, while drained/undrained behavior should be chosen based on permeability and construction duration. - In short: the main concerns are **water pressure, settlement, uplift/base heave, and seepage/piping**, and the design should focus on **proper investigation, groundwater control, adequate embedment, staged construction, and monitoring**. [2][5][7]

**Top citations:** 4.2 Loading Conditions p.[43]; 4.2.3 Water Pressure p.[44, 45]; 8.3.3.2 Consolidation p.[104, 105]

---

## Step 7 — Every recorded lab run

Each lab notebook recorded its runs to `notebooks/results/lab_runs.jsonl`. The table gives a scannable overview (with a longer preview than before); the cell after it prints the **full** recorded answers so no detail is lost.

In [7]:
import pandas as pd

runs = lab.load_runs()
# Keep the most recent entry per label so re-running this notebook does not show duplicates.
runs = list({r['label']: r for r in runs}.values())
print('recorded runs:', len(runs))
if runs:
    rdf = pd.DataFrame([
        {
            'label': r['label'],
            'skill_profile': r['skill_profile'],
            'retrieval_mode': r['retrieval_mode'],
            'scoring_profile': r['scoring_profile'],
            'citations': r['citation_count'],
            'elapsed_s': r['elapsed_s'],
            'answer_chars': len(r['answer']),
            'answer_preview': r['answer'][:240].replace(chr(10), ' ') + ('…' if len(r['answer']) > 240 else ''),
        }
        for r in runs
    ])
    display(rdf)

recorded runs: 19


,label,skill_profile,retrieval_mode,scoring_profile,citations,elapsed_s,answer_chars,answer_preview
0,lab03_baseline_fulltext,baseline_extract,full_text,default,8,1.98,1080,The main objectives of site investigation are ...
1,lab04_chunkvector_full_text,chunk_vector,full_text,default,8,1.65,1045,"However, their locations, length and positioni..."
2,lab04_chunkvector_vector,chunk_vector,vector,default,8,7.37,1029,The presence of loose fill renders a site more...
3,lab04_chunkvector_hybrid,chunk_vector,hybrid,default,8,4.24,962,Loss of overall stability is likely to occur i...
4,lab05_genai_hybrid,genai_enrichment,hybrid,default,8,7.52,1041,"GEO Publication No. 1/2023, **Deep Excavation ..."
5,lab05_genai_agentic,genai_enrichment,agentic,default,8,8.78,2612,- **Key excavation risks** in high groundwater...
6,lab06_visual_hybrid,visual_nlp,hybrid,default,7,7.30,929,"However, this method only calculates the later..."
7,lab06_visual_agentic,visual_nlp,agentic,default,8,17.71,1508,- The figures show that **ground settlement an...
8,lab07_agentic_q2,genai_enrichment,agentic,default,8,8.69,3258,Key excavation risks and design considerations...
9,lab07_hybrid_q2,genai_enrichment,hybrid,default,8,7.50,1041,"GEO Publication No. 1/2023, **Deep Excavation ..."


### Full recorded answers

The same runs, untruncated, so you can read exactly what each profile and mode produced.

In [8]:
from IPython.display import Markdown, display

for r in runs:
    display(Markdown(
        f"#### {r['label']} · `{r['retrieval_mode']}` · profile: {r['skill_profile']} · "
        f"{r['citation_count']} citations\n\n"
        f"**Q:** {r['question']}\n\n"
        f"{r['answer']}"
    ))
    display(Markdown('---'))

#### lab03_baseline_fulltext · `full_text` · profile: baseline_extract · 8 citations

**Q:** What are the objectives of site investigation for ELS works?

The main objectives of site investigation are to acquire knowledge of site characteristics that affect such works and plan for their safe execution, with due consideration given to the nearby buildings/structures/services. When planning ELS works, site investigation should normally include a detailed desk study, site reconnaissance, topographic survey, groun [1]

Supporting evidence:
- The various stages of site investigation, design and construction of ELS works require coordinated input from experienced designers and contractors. Continuous involvement of the designer of the ELS works is essential for verifying both the validity of the assumed ground conditions and the expected performance of the ELS works. [5]
- Good practice for site investigation and selection of geotechnical parameters that are crucial for the design of ELS works and associated key considerations are presented in Chapter 2. A review of common types of excavation support systems and technical considerations for the design and construction of ELS works are discussed in Chapters 3, 4 and 5. [2]

---

#### lab04_chunkvector_full_text · `full_text` · profile: chunk_vector · 8 citations

**Q:** Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?

However, their locations, length and positioning of response zones should be properly planned with due consideration given to the hydrogeological condition of the groundwater regime adjacent to the site. Where the ground conditions comprise stratified marine clay underlain by sandy soils (e.g. [5]

Supporting evidence:
- Good practice for site investigation and selection of geotechnical parameters that are crucial for the design of ELS works and associated key considerations are presented in Chapter 2. A review of common types of excavation support systems and technical considerations for the design and construction of ELS works are discussed in Chapters 3, 4 and 5. [7]
- Given the temporary nature of ELS works, DGWL should be related to possible scenarios that could occur within the duration of the works for different limit states. It is not necessary to consider the effects of long-term and extreme events (e.g. due to climate change). Based on local experience and practice, the following considerations are usually adopted i [6]

---

#### lab04_chunkvector_vector · `vector` · profile: chunk_vector · 8 citations

**Q:** Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?

The presence of loose fill renders a site more susceptible to excessive ground loss during piling operations and bulk excavation, particularly where the groundwater table is near the ground surface. This is commonly the case in reclaimed land where the fill was placed by the hydraulic filling method. [1]

Supporting evidence:
- Site-specific groundwater and drainage conditions should be ascertained within and in the vicinity of the site, and their likely response to, for example, storms, seasonal rise, artesian conditions or tidal action. Field tests normally yield more reliable parameters of the mass permeability of the ground than laboratory tests. [4]
- On the other hand, site supervisory staff should always be alert and take note of any signs of possible ground loss and formation of sinkholes, which typically include the following abnormalities: (a) Larger than expected groundwater discharge seeping into the excavation (e.g. the need to operate more submersible pumps to maintain a dry condition); (b) Signi [3]

...

---

#### compare_hybrid_Q2 · `hybrid` · profile: visual_nlp · 8 citations

**Q:** Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?

GEO Publication No. 1/2023, **Deep Excavation Design and Construction**, is a Hong Kong Geotechnical Engineering Office publication first published in December 2023. Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is. [5]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and should be aimed to determine site-specific  [4]
- As discussed in Section 2.1.2, the differential piezometric pressures between marine clay and alluvial sand layers could lead to porewater pressure dissipation and associated consolidation settlement in the clay layer even when the groundwater level in an overlying fill layer remains stable. [6]

---

#### compare_agentic_Q2 · `agentic` · profile: visual_nlp · 8 citations

**Q:** Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?

For a site with **high groundwater and clay layers**, the main excavation risks and design considerations are: - **Groundwater-related instability and seepage**: the groundwater level must be assessed carefully, because changes in groundwater can affect wall stability and ground deformation. The design groundwater level for ultimate limit state should reflect the highest groundwater level anticipated during the works, based on site monitoring and site-specific hydrogeological conditions. [1][2] - **Consolidation settlement in clay**: clayey, low-permeability layers can undergo consolidation settlement when porewater pressures change. In the cited case, differential piezometric pressure between clay and sand led to dissipation of water from the clay and settlement/tilting of nearby structures. [3][4] - **Uplift / base heave risk**: if low-permeability clay overlies sandy soil and artesian pressure beneath the clay exceeds overburden pressure, uplift can occur; soft clay extending below the excavation base also increases **base heave** risk. [5][6] - **Piping and seepage failure**: upward seepage into the excavation base can reduce effective stress to zero and wash out soil; this is more likely in loose, permeable sandy layers, and deeper wall embedment may be needed to lengthen the seepage path. [7] - **Overall stability issues**: excavations near steeply sloping ground, high groundwater, or weak layers such as soft clay below the wall toe are at greater risk of overall instability. [8] - **Wall deflection / ground settlement from dewatering**: excessive drawdown outside the excavation can cause additional settlement, so drawdown must be controlled and monitored. - **Need for careful seepage analysis and instrumentation**: seepage may be steady-state or hydrostatic depending on soil conditions; in layered ground, simplified assumptions may be inaccurate, so flow-net or numerical seepage analysis and strategically placed piezometers/standpipes are recommended. Key design considerations include: - **Plan groundwater monitoring early** and install standpipes/piezometers at suitable depths, especially where clay overlies permeable layers or confined aquifers may exist. [1] - **Select a water cut-off / retaining system with sufficient embedment**; deeper embedment below soft clay or into firmer ground may be needed for uplift, base heave, and piping resistance. [5][7][6] - **Consider dewatering strategy carefully**: if there is a confined aquifer below clay, dewatering beneath the clay layer or pressure relief holes may be required; in sensitive urban sites, staged excavation with struts is preferred to limit deformation. [5] - **Use realistic groundwater design levels for ULS and SLS**, including sensitivity checks where seasonal variation, tidal influence, or damming effects may matter. [1] - **Address serviceability, not just stability**: groundwater drawdown, wall installation, strut preloading, and support removal are all sources of deformation and should be included in SLS design.[2] - **Monitor and react during construction**: trigger values for groundwater level and settlement should be set, with contingency measures such as regrouting or recharge wells if drawdown or seepage becomes excessive. - **Buildability matters**: the wall type, strutting layout, construction sequence, and handling of clay/sand interfaces should all be designed with construction practicality and safety in mind. If you want, I can turn this into a short checklist for a geotechnical design note or method statement.

---

## How to choose a retrieval mode

| Mode | Best for | Why |
| --- | --- | --- |
| **Full-text** | exact terminology, definitions, standards, tables | lexical BM25 matches explicit wording |
| **Vector** | paraphrased / conceptual questions | embedding similarity finds meaning, not words |
| **Hybrid** | precise + contextual answers | fuses keyword precision with semantic recall |
| **Agentic** | multi-hop reasoning, synthesis across sections | plans subqueries and grounds a cited answer |

Use the **coverage** scores above to back this up with evidence from your own run: the mode that covers the most rubric points with well-targeted citations is the better fit for that question type. And across ingestion profiles, each enrichment layer (chunking → vectors → generative cues → visual/NLP) adds retrievable signal, with the biggest gains on conceptual and diagram-grounded questions.